#### Load Data set: `build_datasets()`

In [ ]:
import os, glob
import numpy as np
DATA_DIR = "./quadrature_data_4qubits"
OUT_X = "dataset_X.txt"
OUT_Y = "dataset_y.txt"
def parse_complex_string(s: str) -> complex:
    """Convert a string like '(1.23+4.56j)' into a Python complex number.
    Parentheses are removed if present. """
    s = s.strip()
    if s.startswith("(") and s.endswith(")"):
        s = s[1:-1]
    return complex(s)
def load_single_file(file_path: str) -> np.ndarray:
    """Load a file such as '0000.txt'.
    Returns a numpy array of shape (4, 1000):
        axis 0 → qubit index (1..4)
        axis 1 → sample index (0..999)"""
    with open(file_path, "r") as f:
        lines = [ln.strip() for ln in f.readlines() if ln.strip()]
    if len(lines) != 4:
        raise ValueError(f"{file_path}: Expected 4 lines, found {len(lines)}.")
    qubit_arrays = []
    for i, line in enumerate(lines):
        tokens = line.split()
        complex_vals = [parse_complex_string(tok) for tok in tokens]
        arr = np.array(complex_vals, dtype=np.complex128)
        if arr.size != 1000:
            raise ValueError(
                f"{file_path},line{i+1}:Expected 1000 values,found{arr.size}.")
        qubit_arrays.append(arr)
    return np.stack(qubit_arrays, axis=0)  # shape (4, 1000)
def bits_from_filename(filename: str) -> np.ndarray:
    """Convert a filename like '0110.txt' into a 4-bit label [q1, q2, q3, q4].
    Bits are read from right to left:
        rightmost bit → qubit1
        leftmost bit → qubit4"""
    base = os.path.splitext(os.path.basename(filename))[0]  # '0110'
    if len(base) != 4 or any(c not in "01" for c in base):
        raise ValueError(f"File name must be a 4-bit string, got '{base}'")
    # Reverse bit order: LSB = qubit1
    bits = [int(b) for b in base[::-1]]
    return np.array(bits, dtype=int)

def build_datasets():
    """Build dataset_X.txt and dataset_y.txt
    dataset_X: 16000 × 4 (complex values)
    dataset_y: 16000 × 4 (bit labels)"""
    pattern = os.path.join(DATA_DIR, "[01][01][01][01].txt")
    file_paths = glob.glob(pattern)
    if len(file_paths) != 16:
        print("WARNING: Expected 16 files, found:", len(file_paths))
    # Sort files from 0000 → 1111
    file_paths = sorted(
        file_paths,
        key=lambda fp: int(os.path.splitext(os.path.basename(fp))[0], 2),)
    all_X_rows = []
    all_y_rows = []
    for fp in file_paths:
        base = os.path.splitext(os.path.basename(fp))[0]
        print(f"Processing file: {base}.txt")
        data_4x1000 = load_single_file(fp)  # shape (4,1000)
        num_qubits, num_samples = data_4x1000.shape
        assert num_qubits == 4
        # Label vector [q1 q2 q3 q4]
        label_bits = bits_from_filename(fp)
        # Prepare blocks of data and labels
        X_block = data_4x1000.T          # shape (1000,4)
        y_block = np.tile(label_bits, (num_samples, 1))  # shape (1000,4)
        all_X_rows.append(X_block)
        all_y_rows.append(y_block)
    # Concatenate all 16 files → 16000 rows
    X = np.vstack(all_X_rows)
    y = np.vstack(all_y_rows)
    print("Final X shape:", X.shape)
    print("Final y shape:", y.shape)
    # Write complex data WITHOUT parentheses
    with open(OUT_X, "w") as f:
        for row in X:
            # Format as: +1.23e+00+4.56e+00j  -3.21e-01+8.76e-01j  ...
            line = " ".join(f"{z.real:+.9e}{z.imag:+.9e}j" for z in row)
            f.write(line + "\n")
    # Write labels as integers
    np.savetxt(OUT_Y, y, fmt="%d")
    print(f"Files '{OUT_X}' and '{OUT_Y}' successfully written.")